# HW06 plotting: functions of time and Fourier partial sums

*HW06 helper notebook, PHY 317.* Run each cell in order with **Shift-Enter**. You only need to edit the cells marked **Your turn**.

Two HW06 problems are marked [Computer]:

- **5.32(c)** (optional): plot an underdamped oscillator released from rest, $x(t) = e^{-\beta t}\left[B_1\cos(\omega_1 t) + B_2 \sin(\omega_1 t)\right]$ with $\omega_1 = \sqrt{\omega_0^2 - \beta^2}$, for several damping constants $\beta$ on one picture.
- **5.49** (the alternative to the square-wave problem): plot the triangle wave of Figure 5.28(a) together with partial sums of its Fourier series,
$$f(t) = \sum_{n=0}^{\infty}\left[a_n\cos(n\omega t) + b_n\sin(n\omega t)\right],$$
like Taylor's Figure 5.23.

The physics (finding $B_1$, $B_2$, $a_n$, $b_n$) is yours, on paper. This notebook shows the two Python tools on *different* functions: plotting a function of $t$ for several parameter values (section 3), and adding up a Fourier series with a `for` loop (section 5). Then it gives you a template for each problem.

## 1. Predict first (before you run anything)

Double-click the cell below this one, type your answers, then Shift-Enter. A sentence each is plenty.

- **5.32:** with $\omega_0 = 1$ rad/s and $\beta = 0.1$ s$^{-1}$, about how many oscillations will you see in the first 20 seconds, and how big will the last ones be compared with the first? Of $\beta = 0.3$ and $\beta = 1$ s$^{-1}$, which one gets close to zero sooner?
- **5.49:** the triangle wave has corners but no jumps, unlike the rectangular pulse in section 5. Will a few Fourier terms imitate it better or worse than they imitate the pulse? Where on the wave will a few-term sum miss the most?

*(your prediction here)*

## 2. Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 3. Plotting a function of time, for several parameter values

Example (not a homework problem): an object dropped from rest with linear air drag (Chapter 2) has velocity
$$v(t) = v_\text{ter}\left(1 - e^{-t/\tau}\right), \qquad v_\text{ter} = g\tau .$$
Here it is for four values of the time constant $\tau$ on one set of axes:

In [ ]:
g = 9.8   # m/s^2


def v_drag(t, tau):
    v_ter = g * tau
    return v_ter * (1 - np.exp(-t / tau))


t = np.linspace(0, 10, 500)    # 500 times from 0 to 10 seconds
taus = [0.5, 1.0, 2.0, 5.0]    # seconds

plt.figure(figsize=(8, 4))
for tau in taus:
    plt.plot(t, v_drag(t, tau), label=f"tau = {tau} s")
plt.xlabel("t (seconds)")
plt.ylabel("v (m/s)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Line by line:

- `np.linspace(0, 10, 500)` makes an array of 500 evenly spaced times from 0 to 10 s.
- `v_drag(t, tau)` takes that whole array at once: `np.exp` works on every entry, so the function hands back 500 velocities. No loop over times is needed.
- `taus = [0.5, 1.0, 2.0, 5.0]` is a Python *list*. `for tau in taus:` runs the indented line once for each entry, with `tau` set to that entry. Each pass draws one curve.
- `label=f"tau = {tau} s"` names each curve; the `f` in front lets Python fill in `{tau}`. `plt.legend()` shows the names.
- Anything not indented under the `for` (the axis labels, `plt.show()`) runs once, after all the curves are drawn.

## 4. Your turn: Problem 5.32(c)

Taylor's numbers: $x_0 = 1$, $\omega_0 = 1$ rad/s, $0 < t < 20$ s, and $\beta = 0, 0.02, 0.1, 0.3, 1$ s$^{-1}$.

Fill in `x_532` with your answer to part (a). One catch: at $\beta = 1 = \omega_0$, $\omega_1 = 0$, and a formula with $\omega_1$ in a denominator gives `nan` ("not a number") and a warning. For that one value use your critical-damping answer from part (b). The `if` does the sorting for you: when `beta == omega0` the function returns from inside the `if` and never reaches the last line.

In [ ]:
x0 = 1.0       # starting position (released from rest)
omega0 = 1.0   # rad/s


def x_532(t, beta):
    omega1 = np.sqrt(omega0**2 - beta**2)
    if beta == omega0:
        # Change the line below.
        return 0.0 * t
    # Change the line below.
    return 0.0 * t

Now plot all five on one set of axes, the way section 3 did:

In [ ]:
t = np.linspace(0, 20, 1000)   # seconds
betas = [0, 0.02, 0.1, 0.3, 1]  # 1/s

plt.figure(figsize=(9, 4))
for beta in betas:
    plt.plot(t, x_532(t, beta), label=f"beta = {beta} 1/s")
plt.xlabel("t (seconds)")
plt.ylabel("x")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))   # legend outside, to the right
plt.grid(alpha=0.3)
plt.show()

## 5. Adding up a Fourier series with a `for` loop

Example (not a homework problem): Taylor's Example 5.4, the periodic rectangular pulse of Figure 5.22, with period $\tau = 1$ s, pulse duration $\Delta\tau = 0.25$ s and height $f_\text{max} = 1$. Taylor finds (5.86)-(5.87):
$$a_0 = f_\text{max}\frac{\Delta\tau}{\tau}, \qquad a_n = \frac{2 f_\text{max}}{n\pi}\sin\left(\frac{n\pi\Delta\tau}{\tau}\right) \ (n \ge 1), \qquad b_n = 0 .$$
The code below rebuilds his Figure 5.23: the pulse with the sum of the first 3 terms, and with the first 11.

(Heads-up if you compare numbers with the book: the decimals Taylor prints in (5.89), 0.24, 0.23, 0.20, ..., are not what (5.87) gives for $\Delta\tau = 0.25$ s. The code prints the coefficients from (5.87), 0.45, 0.32, 0.15, ..., and those are the ones that reproduce his Figure 5.23.)

In [ ]:
tau_demo = 1.0     # s, period
dtau = 0.25        # s, pulse duration
fmax = 1.0
omega_demo = 2 * np.pi / tau_demo


def pulse(t):
    t_in_period = np.mod(t + tau_demo / 2, tau_demo) - tau_demo / 2
    return np.where(np.abs(t_in_period) < dtau / 2, fmax, 0.0)


def a_demo(n):
    if n == 0:
        return fmax * dtau / tau_demo
    return 2 * fmax / (n * np.pi) * np.sin(n * np.pi * dtau / tau_demo)


def b_demo(n):
    return 0.0


def partial_sum_demo(t, n_terms):
    total = a_demo(0) * np.ones_like(t)
    for n in range(1, n_terms):
        total = total + a_demo(n) * np.cos(n * omega_demo * t) + b_demo(n) * np.sin(n * omega_demo * t)
    return total


for n in range(7):
    print(n, round(a_demo(n), 3))

Line by line:

- `pulse` is the function itself, so we can draw it under the sums. `np.mod(t + tau_demo/2, tau_demo) - tau_demo/2` moves every time into one period, $-\tau/2 \le t < \tau/2$, which is how one formula makes a function repeat. `np.where(condition, A, B)` gives `A` where the condition is true and `B` where it is false.
- `a_demo(n)` and `b_demo(n)` are the coefficients. The `if n == 0:` handles $a_0$, which has its own formula; for every other `n` the function skips the `if` and reaches the last line.
- `partial_sum_demo(t, n_terms)` starts the running total at the constant $a_0$ (`np.ones_like(t)` is an array of ones the same size as `t`), then `for n in range(1, n_terms):` adds the $n$th cosine and sine, for $n = 1, 2, \ldots$, up to `n_terms - 1`. So `n_terms = 3` means $a_0$ plus $n = 1$ and $n = 2$: Taylor's "first three terms".
- `total = total + ...` is the running sum: each pass through the loop adds one more term to what was already there.

In [ ]:
t = np.linspace(-1.5, 1.5, 2000)   # seconds

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, n_terms in zip(axes, [3, 11]):
    ax.plot(t, pulse(t), "k", lw=1)
    ax.plot(t, partial_sum_demo(t, n_terms))
    ax.set_title(f"f(t) (black) and the first {n_terms} terms")
    ax.set_xlabel("t (seconds)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("f")
plt.show()

New here: `plt.subplots(1, 2)` makes two panels side by side (`sharey=True` gives them the same vertical scale), and `zip(axes, [3, 11])` pairs the first panel with 3 terms and the second with 11, so the `for` loop draws one panel per pass. Compare with Taylor's Figure 5.23.

## 6. Your turn: Problem 5.49

The triangle wave of Figure 5.28(a): period $\tau = 2$ s, $f_\text{max} = 1$. Fill in, the way section 5 did:

1. `f_549`: the function itself on one period. The first line, already written, moves $t$ into $-1 \le t < 1$; you write $f$ there (read it off Figure 5.28(a)).
2. `a_549` and `b_549`: your coefficients from part (a), including $a_0$.

`partial_sum_549` is section 5's loop with the names changed; you don't need to edit it. As handed out every function returns zero, so the plot below shows flat lines.

In [ ]:
tau_549 = 2.0    # s, period
fmax_549 = 1.0
omega_549 = 2 * np.pi / tau_549


def f_549(t):
    t_in_period = np.mod(t + tau_549 / 2, tau_549) - tau_549 / 2
    # Change the line below.
    return 0.0 * t_in_period


def a_549(n):
    if n == 0:
        # Change the line below.
        return 0.0
    # Change the line below.
    return 0.0


def b_549(n):
    # Change the line below.
    return 0.0


def partial_sum_549(t, n_terms):
    total = a_549(0) * np.ones_like(t)
    for n in range(1, n_terms):
        total = total + a_549(n) * np.cos(n * omega_549 * t) + b_549(n) * np.sin(n * omega_549 * t)
    return total

The plot: like Figure 5.23, the function with a couple of terms, and with six or so. Change the numbers in `n_terms_list` if you want to see others.

In [ ]:
t = np.linspace(-3, 3, 2000)   # seconds
n_terms_list = [2, 6]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, n_terms in zip(axes, n_terms_list):
    ax.plot(t, f_549(t), "k", lw=1)
    ax.plot(t, partial_sum_549(t, n_terms))
    ax.set_title(f"f(t) (black) and the first {n_terms} terms")
    ax.set_xlabel("t (seconds)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("f")
plt.show()

## 7. What you found

Double-click each answer cell, type, Shift-Enter.

**a. 5.32(c), optional.** Describe the five curves: which ones oscillate, and how does the damping change the picture as $\beta$ grows? Compare with your prediction.

*(your answer here)*

**b. 5.49.** How well do two terms imitate the triangle wave, and how well do six? Where is the sum worst? Compare with your prediction.

*(your answer here)*

**c. Assessment.** One check on your plots (for example: what should every partial sum give at a particular $t$, or what should the $\beta = 0$ curve be?).

*(your answer here)*

**d. What didn't work** the first time, if anything.

Then put your plots (screenshots are fine) and these answers in your homework PDF.

*(your answer here)*